## Without web search

In [8]:
from dotenv import load_dotenv

load_dotenv()

True

In [9]:
from langchain.agents import create_agent
from langchain.chat_models import init_chat_model

model = init_chat_model(model="llama-3.1-8b-instant", model_provider="groq" , temperature=0)

agent = create_agent(
    model=model
)

In [10]:
from langchain.messages import HumanMessage

question = HumanMessage(content="How up to date is your training knowledge?")

response = agent.invoke(
    {"messages": [question]}
)

In [11]:
print(response['messages'][-1].content)

My training data is current up to 2023. I do not have real-time access to information and may not always have the most recent developments or updates on specific topics.


## Add web search tool

In [12]:
from langchain.tools import tool
from typing import Dict, Any
from dotenv import load_dotenv
load_dotenv()

from tavily import TavilyClient
import os

tavily_client = TavilyClient()
@tool
def web_search(query: str) -> Dict[str, Any]:

    """Search the web for information"""

    return tavily_client.search(query)

web_search.invoke("Who is the current mayor of San Francisco?")

{'query': 'Who is the current mayor of San Francisco?',
 'follow_up_questions': None,
 'answer': None,
 'images': [],
 'results': [{'url': 'https://en.wikipedia.org/wiki/Mayor_of_San_Francisco',
   'title': 'Mayor of San Francisco - Wikipedia',
   'content': 'The current mayor is Democrat Daniel Lurie.',
   'score': 0.95324475,
   'raw_content': None},
  {'url': 'https://apnews.com/article/san-francisco-new-mayor-liberal-city-81ea0a7b37af6cbb68aea7ef5cc6a4f0',
   'title': "San Francisco's new mayor is starting to unite the fractured city",
   'content': 'San Francisco Mayor Daniel Lurie, a political newcomer and Levi Strauss heir, has marked his first 100 days with a hands-on, business-friendly approach.',
   'score': 0.8755427,
   'raw_content': None},
  {'url': 'https://www.sf.gov/departments--office-mayor',
   'title': 'Office of the Mayor - SF.gov',
   'content': 'Daniel Lurie is the 46th Mayor of the City and County of San Francisco.',
   'score': 0.8488849,
   'raw_content': None

In [13]:
agent = create_agent(
    model=model,
    tools=[web_search]
)

question = HumanMessage(content="Who is the current mayor of San Francisco?")

response = agent.invoke(
    {"messages": [question]}
)

In [14]:
from pprint import pprint

pprint(response['messages'])

[HumanMessage(content='Who is the current mayor of San Francisco?', additional_kwargs={}, response_metadata={}, id='a711d334-c68a-4bbf-a50b-87bda86efad0'),
 AIMessage(content='<web_search>{"query": "current mayor of San Francisco"};</web_search>', additional_kwargs={}, response_metadata={'token_usage': {'completion_tokens': 18, 'prompt_tokens': 219, 'total_tokens': 237, 'completion_time': 0.024527694, 'completion_tokens_details': None, 'prompt_time': 0.013235191, 'prompt_tokens_details': None, 'queue_time': 0.045498859, 'total_time': 0.037762885}, 'model_name': 'llama-3.1-8b-instant', 'system_fingerprint': 'fp_f757f4b0bf', 'service_tier': 'on_demand', 'finish_reason': 'stop', 'logprobs': None, 'model_provider': 'groq'}, id='lc_run--019d3f2a-3788-71b2-910e-2474cbfa8629-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 219, 'output_tokens': 18, 'total_tokens': 237})]


trace: https://smith.langchain.com/public/59432173-0dd6-49e8-9964-b16be6048426/r